# Task 8: Smart Security System with Boundary Detection

**Lab 06 solution**  
Run cells from top to bottom. Change only the paths in the configuration cell when using your own media.

In [4]:
# Install once if required:
# !pip install opencv-contrib-python numpy matplotlib PyWavelets
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def show(img, title='', cmap=None, size=(12,7)):
    plt.figure(figsize=size)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

## Objective
Monitor a predefined security zone in a video. Background subtraction detects newly introduced objects, contour boundaries localize them, and intersection with the security zone triggers an alarm annotation.

In [6]:
# ============================================================
# Initialize Security Video + Motion Detection
# ============================================================

VIDEO_PATH = 'security_video.mp4'
OUTPUT_PATH = 'task8_security_output.mp4'


# ============================================================
# Open Security Video
# ============================================================

cap = cv.VideoCapture(
    VIDEO_PATH
)

if not cap.isOpened():

    raise FileNotFoundError(
        f"Video not found or could not be opened: {VIDEO_PATH}"
    )


# ============================================================
# Get Video Properties
# ============================================================

fps = cap.get(
    cv.CAP_PROP_FPS
)

if fps <= 0:
    fps = 25


W = int(
    cap.get(
        cv.CAP_PROP_FRAME_WIDTH
    )
)

H = int(
    cap.get(
        cv.CAP_PROP_FRAME_HEIGHT
    )
)

# ============================================================
# Initialize Output Video
# ============================================================

fourcc = cv.VideoWriter_fourcc(
    *'mp4v'
)

out = cv.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (W, H)
)

if not out.isOpened():

    raise RuntimeError(
        f"Could not create output video: {OUTPUT_PATH}"
    )


# ============================================================
# Define Sensitive Zone
# ============================================================

# Central-lower rectangular region.
# Modify these fractions if the sensitive area changes.

zx1 = int(0.30 * W)
zy1 = int(0.35 * H)

zx2 = int(0.72 * W)
zy2 = int(0.90 * H)


# ============================================================
# Initialize Background Subtractor
# ============================================================

sub = cv.createBackgroundSubtractorMOG2(
    history=300,
    varThreshold=35,
    detectShadows=True
)


# ============================================================
# Morphological Processing Kernel
# ============================================================

k = np.ones(
    (5, 5),
    np.uint8
)


# ============================================================
# Tracking Counters
# ============================================================

alarm_frames = 0
frame_count = 0


# ============================================================
# Display Configuration
# ============================================================

print(
    'Video resolution:',
    f'{W} x {H}'
)

print(
    'Video FPS:',
    round(fps, 2)
)

print(
    'Sensitive zone:',
    (zx1, zy1, zx2, zy2)
)

print(
    'Output:',
    OUTPUT_PATH
)


Video resolution: 640 x 480
Video FPS: 25.0
Sensitive zone: (192, 168, 460, 432)
Output: task8_security_output.mp4


In [7]:
# ============================================================
# Security Zone Monitoring
# ============================================================

try:

    while True:

        # ----------------------------------------------------
        # Read next video frame
        # ----------------------------------------------------

        ok, frame = cap.read()

        if not ok:
            break

        frame_count += 1


        # ----------------------------------------------------
        # Apply background subtraction
        # ----------------------------------------------------

        fg = sub.apply(frame)

        # Remove MOG2 shadow pixels
        fg[fg == 127] = 0


        # ----------------------------------------------------
        # Clean foreground mask
        # ----------------------------------------------------

        fg = cv.morphologyEx(
            fg,
            cv.MORPH_OPEN,
            k,
            iterations=1
        )

        fg = cv.morphologyEx(
            fg,
            cv.MORPH_CLOSE,
            k,
            iterations=2
        )


        # ----------------------------------------------------
        # Find moving-object contours
        # ----------------------------------------------------

        contours, _ = cv.findContours(
            fg,
            cv.RETR_EXTERNAL,
            cv.CHAIN_APPROX_SIMPLE
        )

        alarm = False


        # ----------------------------------------------------
        # Check detected objects
        # ----------------------------------------------------

        for c in contours:

            # Ignore small movements/noise
            if cv.contourArea(c) < 1200:
                continue

            x, y, w, h = cv.boundingRect(c)


            # ------------------------------------------------
            # Calculate intersection with security zone
            # ------------------------------------------------

            intersection_width = max(
                0,
                min(x + w, zx2)
                - max(x, zx1)
            )

            intersection_height = max(
                0,
                min(y + h, zy2)
                - max(y, zy1)
            )

            intersection_area = (
                intersection_width
                * intersection_height
            )


            # ------------------------------------------------
            # Object entered the security zone
            # ------------------------------------------------

            if intersection_area > 0:

                alarm = True

                cv.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 0, 255),
                    3
                )

                cv.drawContours(
                    frame,
                    [c],
                    -1,
                    (0, 255, 255),
                    2
                )


        # ====================================================
        # Draw Security Zone
        # ====================================================

        cv.rectangle(
            frame,
            (zx1, zy1),
            (zx2, zy2),
            (255, 0, 0),
            3
        )


        # ====================================================
        # Display Security Status
        # ====================================================

        if alarm:

            alarm_frames += 1

            cv.putText(
                frame,
                'ALARM: UNAUTHORIZED OBJECT IN SECURITY ZONE',
                (20, 45),
                cv.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                3
            )

        else:

            cv.putText(
                frame,
                'SECURITY ZONE CLEAR',
                (20, 45),
                cv.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )


        # ----------------------------------------------------
        # Write processed frame
        # ----------------------------------------------------

        out.write(frame)


finally:

    # Always release video resources
    cap.release()
    out.release()


# ============================================================
# Final Results
# ============================================================

print(
    f'Saved {OUTPUT_PATH}'
)

print(
    f'Alarm active for '
    f'{alarm_frames}/{frame_count} frames'
)

if frame_count > 0:

    alarm_rate = (
        alarm_frames
        / frame_count
        * 100
    )

    print(
        f'Alarm frame percentage: '
        f'{alarm_rate:.1f}%'
    )


Saved task8_security_output.mp4
Alarm active for 51/90 frames
Alarm frame percentage: 56.7%


## Reliability notes
The first frames train the background model, so begin with an empty scene. The area threshold rejects small motion/noise. MOG2 shadow labels are removed. Camera movement, sudden lighting changes, and persistent crowds can cause false alarms; a fixed camera and a short initialization period improve reliability.